# GRU dual-head — the next ladder step (MPS)

One shared GRU encoder over the **past 6h** (24 × 15-min steps) with two heads:

- **Risk head**: 9 logits — P(≥5/15/30cm) × (1h/3h/6h)
- **Depth head**: 15 outputs — P05/P25/P50/P75/P95 × (1h/3h/6h), pinball loss

Implemented in plain PyTorch rather than Darts: Darts' RNN models do probabilistic *regression* well, but a joint classification+quantile dual-head in one network isn't natively supported — plain PyTorch keeps the architecture honest and identical to the roadmap's intent. Runs on Apple **MPS** (float32 throughout).

**The rule: this model is kept ONLY if it beats LightGBM's val PR-AUC on ge15_3h and ge15_6h.** Otherwise LightGBM ships and this notebook becomes an appendix in the report.

In [1]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
torch.manual_seed(42)
np.random.seed(42)

device: mps


## Configuration

In [2]:
TRAINING_DIR = Path("../data/training")
ARTIFACTS = Path("../models/artifacts")

SEQ_LEN = 24                # 6h of 15-min anchors
HORIZONS = [1, 3, 6]
TIERS = [5, 15, 30]
QUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95]
VALID_MIN = 0.8

# per-timestep dynamic features fed to the GRU (NaN -> 0 + indicator)
SEQ_FEATURES = ["fl_depth_now", "fl_max3h", "fl_mean1h",
                "rain_rf1hr_mean", "rain_rf1hr_max", "rain_rf3hr_mean",
                "rain_rf24hr_mean", "rain_rf1hr_delta1h",
                "water_rise1h_mean", "water_rising_share",
                "flow_mean", "flow_negative_share",
                "cal_hour_sin", "cal_hour_cos", "cal_monsoon"]

NEG_FRAC = 0.02             # negatives sampled per epoch (all positives kept)
BATCH = 1024
EPOCHS = 10
LR = 1e-3
HIDDEN = 64
EMB = 8
POS_WEIGHT = 20.0
DEPTH_LOSS_SCALE = 0.05     # balances cm-scale pinball vs BCE
PATIENCE = 2                # early stop on val PR-AUC (ge15_6h)

Y_COLS = [f"y_ge{t}_{h}h" for h in HORIZONS for t in TIERS]      # 9
D_COLS = [f"y_maxdepth_{h}h" for h in HORIZONS]                  # 3
V_COLS = [f"y_valid_{h}h" for h in HORIZONS]                     # 3

## Load splits into flat per-station arrays

Sequences are materialized lazily (index + slice), never as a 17M×24×F tensor.

In [3]:
def load_arrays(name, station_index=None):
    cols = (["station_code", "site_timestamp"] + SEQ_FEATURES
            + Y_COLS + D_COLS + V_COLS)
    table = pq.read_table(TRAINING_DIR / f"{name}.parquet", columns=cols)
    schema = pa.schema([pa.field(f.name, pa.float32())
                        if f.type == pa.float64() else f for f in table.schema])
    df = (table.cast(schema).to_pandas(self_destruct=True)
          .sort_values(["station_code", "site_timestamp"]))

    X = df[SEQ_FEATURES].to_numpy(np.float32)
    miss = np.isnan(X[:, :2]).astype(np.float32)     # depth & max3h indicators
    X = np.concatenate([np.nan_to_num(X), miss], axis=1)
    Y = np.nan_to_num(df[Y_COLS].to_numpy(np.float32))
    D = df[D_COLS].to_numpy(np.float32)              # NaN allowed (masked)
    V = np.nan_to_num(df[V_COLS].to_numpy(np.float32))
    ts = df["site_timestamp"].astype("int64").to_numpy() // 10**9

    codes = df["station_code"].astype(str).to_numpy()
    if station_index is None:
        station_index = {c: i + 1 for i, c in enumerate(np.unique(codes))}  # 0 = unknown
    sid = np.array([station_index.get(c, 0) for c in codes], dtype=np.int64)

    # valid anchors: full lookback within one station, contiguous 15-min grid
    ok = np.zeros(len(df), dtype=bool)
    ok[SEQ_LEN - 1:] = (
        (codes[SEQ_LEN - 1:] == codes[:-(SEQ_LEN - 1)])
        & (ts[SEQ_LEN - 1:] - ts[:-(SEQ_LEN - 1)] == (SEQ_LEN - 1) * 900))
    anchors = np.where(ok)[0]
    print(f"{name}: {len(df):,} rows -> {len(anchors):,} valid anchors, "
          f"{len(station_index)} stations")
    return dict(X=X, Y=Y, D=D, V=V, sid=sid, anchors=anchors,
                station_index=station_index)

train_a = load_arrays("train")
val_a = load_arrays("val", train_a["station_index"])
N_FEAT = train_a["X"].shape[1]
N_STATIONS = len(train_a["station_index"]) + 1

train: 17,250,919 rows -> 17,247,675 valid anchors, 107 stations
val: 3,451,352 rows -> 3,448,862 valid anchors, 107 stations


## Dataset (all positives + sampled negatives per split)

In [4]:
class SeqDataset(Dataset):
    def __init__(self, arrs, neg_frac=None):
        self.a = arrs
        idx = arrs["anchors"]
        if neg_frac is not None:
            pos = idx[arrs["Y"][idx].max(axis=1) > 0]
            neg = idx[arrs["Y"][idx].max(axis=1) == 0]
            keep = np.random.default_rng(42).choice(
                len(neg), int(len(neg) * neg_frac), replace=False)
            idx = np.sort(np.concatenate([pos, neg[keep]]))
            print(f"dataset: {len(pos):,} positives + {len(keep):,} negatives")
        self.idx = idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        seq = self.a["X"][j - SEQ_LEN + 1: j + 1]
        return (torch.from_numpy(seq.copy()),
                torch.tensor(self.a["sid"][j]),
                torch.from_numpy(self.a["Y"][j].copy()),
                torch.from_numpy(np.nan_to_num(self.a["D"][j], nan=-1.0).copy()),
                torch.from_numpy(self.a["V"][j].copy()))

train_ds = SeqDataset(train_a, NEG_FRAC)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=0, drop_last=True)

dataset: 71,395 positives + 343,525 negatives


## Model

In [5]:
class DualHeadGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(N_STATIONS, EMB)
        self.gru = nn.GRU(N_FEAT + EMB, HIDDEN, num_layers=1, batch_first=True)
        self.norm = nn.LayerNorm(HIDDEN)
        self.head_cls = nn.Sequential(nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                      nn.Linear(HIDDEN, len(Y_COLS)))
        self.head_reg = nn.Sequential(nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                      nn.Linear(HIDDEN,
                                                len(HORIZONS) * len(QUANTILES)))

    def forward(self, seq, sid):
        e = self.emb(sid)[:, None, :].expand(-1, seq.shape[1], -1)
        h, _ = self.gru(torch.cat([seq, e], dim=2))
        z = self.norm(h[:, -1])
        logits = self.head_cls(z)                                   # (B, 9)
        depths = nn.functional.softplus(self.head_reg(z))           # (B, 15) >= 0
        return logits, depths.view(-1, len(HORIZONS), len(QUANTILES))

model = DualHeadGRU().to(DEVICE)
print(sum(p.numel() for p in model.parameters()), "parameters")

28344 parameters


## Loss: weighted BCE (risk) + masked pinball (depth)

In [ ]:
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))
QT = torch.tensor(QUANTILES, device=DEVICE)             # (5,)

def step_loss(logits, depths, y, d, v):
    loss_cls = bce(logits, y)
    # depth mask: label observed and window valid enough
    mask = ((d >= 0) & (v >= VALID_MIN)).float()        # (B, 3)
    diff = d.unsqueeze(2) - depths                      # (B, 3, 5)
    pin = torch.maximum(QT * diff, (QT - 1) * diff)     # (B, 3, 5)
    loss_reg = (pin * mask).sum() / mask.sum().clamp(min=1)
    return loss_cls + DEPTH_LOSS_SCALE * loss_reg, loss_cls, loss_reg

## Validation inference helper (batched over all anchors)

In [7]:
@torch.no_grad()
def predict_all(arrs, batch=8192):
    model.eval()
    idx = arrs["anchors"]
    probs = np.empty((len(idx), len(Y_COLS)), np.float32)
    for s in range(0, len(idx), batch):
        j = idx[s: s + batch]
        seq = np.stack([arrs["X"][k - SEQ_LEN + 1: k + 1] for k in j])
        logits, _ = model(torch.from_numpy(seq).to(DEVICE),
                          torch.from_numpy(arrs["sid"][j]).to(DEVICE))
        probs[s: s + batch] = torch.sigmoid(logits).cpu().numpy()
    return idx, probs

def val_prauc(arrs, probs, idx):
    out = {}
    for c, col in enumerate(Y_COLS):
        h = int(col.split("_")[-1][:-1])
        vmask = arrs["V"][idx][:, HORIZONS.index(h)] >= VALID_MIN
        y = arrs["Y"][idx][vmask, c]
        if y.sum() > 0:
            out[col] = float(average_precision_score(y, probs[vmask, c]))
    return out

## Train (early stop on val ge15_6h PR-AUC)

In [8]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
best_metric, best_state, bad = -1.0, None, 0
history = []

for epoch in range(EPOCHS):
    model.train()
    t0, tot = time.time(), 0.0
    for seq, sid, y, d, v in train_dl:
        seq, sid = seq.to(DEVICE), sid.to(DEVICE)
        y, d, v = y.to(DEVICE), d.to(DEVICE), v.to(DEVICE)
        opt.zero_grad()
        loss, lc, lr_ = step_loss(*model(seq, sid), y, d, v)
        loss.backward()
        opt.step()
        tot += float(loss)
    idx, probs = predict_all(val_a)
    pr = val_prauc(val_a, probs, idx)
    metric = pr.get("y_ge15_6h", 0.0)
    history.append({"epoch": epoch, "train_loss": tot / len(train_dl), **pr})
    print(f"epoch {epoch}: loss {tot/len(train_dl):.4f} | "
          f"ge15_3h {pr.get('y_ge15_3h', 0):.4f} | ge15_6h {metric:.4f} | "
          f"{time.time()-t0:.0f}s")
    if metric > best_metric:
        best_metric, bad = metric, 0
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE:
            print("early stop")
            break

model.load_state_dict(best_state)
pd.DataFrame(history)

/var/folders/j1/4ycrk7cj67s3kxcjbx2fknpr0000gn/T/ipykernel_10264/2792141721.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  tot += float(loss)


epoch 0: loss 0.6769 | ge15_3h 0.1734 | ge15_6h 0.0932 | 31s
epoch 1: loss 0.5419 | ge15_3h 0.1518 | ge15_6h 0.0806 | 29s
epoch 2: loss 0.4972 | ge15_3h 0.1951 | ge15_6h 0.1015 | 29s
epoch 3: loss 0.4682 | ge15_3h 0.1459 | ge15_6h 0.0780 | 31s
epoch 4: loss 0.4430 | ge15_3h 0.1754 | ge15_6h 0.0937 | 35s
early stop


,epoch,train_loss,y_ge5_1h,y_ge15_1h,y_ge30_1h,y_ge5_3h,y_ge15_3h,y_ge30_3h,y_ge5_6h,y_ge15_6h,y_ge30_6h
0,0,0.676897,0.354544,0.424097,0.400590,0.172560,0.173353,0.166438,0.099847,0.093168,0.082547
1,1,0.541858,0.399316,0.401714,0.333992,0.200498,0.151805,0.109190,0.116454,0.080630,0.048416
2,2,0.497181,0.441657,0.461559,0.380609,0.218134,0.195053,0.144177,0.132020,0.101502,0.072110
3,3,0.468157,0.433095,0.386700,0.241438,0.213866,0.145879,0.103693,0.127003,0.077974,0.055870
4,4,0.443022,0.448064,0.427404,0.271948,0.220089,0.175399,0.112193,0.132300,0.093733,0.062502


## Verdict: GRU vs LightGBM (val PR-AUC)

In [9]:
idx, probs = predict_all(val_a)
gru_pr = val_prauc(val_a, probs, idx)

lgbm = json.loads((ARTIFACTS / "metrics.json").read_text())["classifiers"]
rows = []
for col, v in gru_pr.items():
    key = col[2:]                      # y_ge15_6h -> ge15_6h
    rows.append({"target": key, "gru_pr_auc": round(v, 4),
                 "lgbm_pr_auc": lgbm.get(key, {}).get("pr_auc")})
cmp = pd.DataFrame(rows)
cmp["winner"] = np.where(cmp.gru_pr_auc > cmp.lgbm_pr_auc, "GRU", "LightGBM")

KEEP_GRU = bool(
    (cmp.set_index("target").loc["ge15_3h", "gru_pr_auc"]
     > cmp.set_index("target").loc["ge15_3h", "lgbm_pr_auc"])
    and (cmp.set_index("target").loc["ge15_6h", "gru_pr_auc"]
         > cmp.set_index("target").loc["ge15_6h", "lgbm_pr_auc"]))
print("KEEP_GRU =", KEEP_GRU)
cmp

KEEP_GRU = False


,target,gru_pr_auc,lgbm_pr_auc,winner
0,ge5_1h,0.4417,0.4671,LightGBM
1,ge15_1h,0.4616,0.4179,GRU
2,ge30_1h,0.3806,0.0002,GRU
3,ge5_3h,0.2181,0.2198,LightGBM
4,ge15_3h,0.1951,0.1774,GRU
5,ge30_3h,0.1442,0.0084,GRU
6,ge5_6h,0.1320,0.1272,GRU
7,ge15_6h,0.1015,0.1018,LightGBM
8,ge30_6h,0.0721,0.0868,LightGBM


## Save artifacts

In [10]:
torch.save({"state_dict": model.state_dict(),
            "station_index": train_a["station_index"],
            "config": {"SEQ_LEN": SEQ_LEN, "SEQ_FEATURES": SEQ_FEATURES,
                       "HIDDEN": HIDDEN, "EMB": EMB,
                       "HORIZONS": HORIZONS, "TIERS": TIERS,
                       "QUANTILES": QUANTILES}},
           ARTIFACTS / "gru_model.pt")
(ARTIFACTS / "gru_metrics.json").write_text(json.dumps(
    {"KEEP_GRU": KEEP_GRU, "val_pr_auc": gru_pr,
     "comparison": cmp.to_dict(orient="records"),
     "history": history}, indent=2))
print("saved gru_model.pt + gru_metrics.json | KEEP_GRU =", KEEP_GRU)

saved gru_model.pt + gru_metrics.json | KEEP_GRU = False


## Decision

- **KEEP_GRU = True** → set `WINNER = "gru"` in `final_test.ipynb`
- **KEEP_GRU = False** → LightGBM ships; report this notebook as the honest negative result (the ladder principle working as intended)

If MPS throws an unsupported-op error, rerun with `PYTORCH_ENABLE_MPS_FALLBACK=1 jupyter lab` — GRU layers themselves are well-supported and stay on-GPU.